# Independent Flat Models: 6 B Algorithms vs 18 Other Algorithms

This notebook trains two independent flat algorithm selectors on the same original data:

- **B6**: the six algorithms ending in `-B-bnb`.
- **Other18**: all remaining 18 algorithms.

It follows the existing leakage-safe flat/Scheme-4 workflow: original-row splitting first, training-only tie expansion, unexpanded held-out test rows, `strat_y` and `strat_y_K`, RF and XGB, test/entire metrics, tolerance analysis, important failures, failure enrichment, and old-rule Top-K combinations.

## Tie handling
Ties are resolved **inside each independent family**. If several algorithms in that family are tied, the original instance is split only once; if it enters training, its tied labels are expanded with divided sample weight. Test rows are never expanded.


In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

REPO_ROOT=Path.cwd()
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))

from restructured_pipeline.src.features import (
    ALL_ALGOS, ROUTER_CLASSB_ALGOS, load_excel_data,
    process_kcore_list_to_statistics, post_process_data,
)
from restructured_pipeline.src.data_splits import _train_test_split_k_aware
from restructured_pipeline.src.train import load_yaml_config
from restructured_pipeline.src.compare_wrong_predictions import filter_tolerance_sheet_for_important_failures
from restructured_pipeline.src.metrics import evaluate_with_tolerance, evaluate_time_metrics_with_tol
from restructured_pipeline.src.evaluate import compute_top_k_metrics

PROJECT_CONFIG=Path("restructured_pipeline/configs/project.yaml")
MODEL_CONFIG=Path("restructured_pipeline/configs/models_scheme1.yaml")
project=load_yaml_config(PROJECT_CONFIG)
model_cfg=load_yaml_config(MODEL_CONFIG)

FEATURE_COLS=list(project["features"]["feature_cols"])
B6_ALGOS=list(ROUTER_CLASSB_ALGOS)
OTHER18_ALGOS=[a for a in ALL_ALGOS if a not in B6_ALGOS]
FAMILIES={"B6":B6_ALGOS,"Other18":OTHER18_ALGOS}
SPLITS=("strat_y","strat_y_K"); MODELS=("RF","XGB"); TOLERANCES=(0.0,0.1,0.2,0.5)
TEST_SIZE=project["split_rules"]["defaults"]["test_size"]
RANDOM_STATE=project["split_rules"]["defaults"]["random_state"]
WEIGHT_COL=project["split_rules"]["defaults"]["weight_col"]
TIE_TOLERANCE=project["split_rules"]["defaults"]["tie_tolerance"]
EPSILON=project["evaluation"]["epsilon"]
OUTPUT_DIR=Path("restructured_pipeline/outputs/reports/B6_vs_Other18_independent_flat")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

assert len(B6_ALGOS)==6 and len(OTHER18_ALGOS)==18
assert set(B6_ALGOS).isdisjoint(OTHER18_ALGOS)
print("B6:",B6_ALGOS)
print("Other18:",OTHER18_ALGOS)


## 1. Load and preprocess the canonical dataset
Uses the existing `features.py` loader and feature-engineering functions; no alternate input dataset is created.


In [ ]:
data_cfg=project["data"]
df=load_excel_data(
    data_cfg["file_path"], timeout_replacement=data_cfg["timeout_replacement"],
    y_cols=data_cfg["raw_target_columns"], sep=data_cfg["sep"],
    quotechar=data_cfg["quotechar"], engine=data_cfg["engine"],
    rename_columns=data_cfg["rename_columns"],
)
if "kcore_list" in df.columns and any(c not in df.columns for c in FEATURE_COLS):
    process_kcore_list_to_statistics(df,keep_col=False)
df=post_process_data(df)
missing=[c for c in ["dataset","k_value",*FEATURE_COLS,*ALL_ALGOS] if c not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}")
if WEIGHT_COL not in df.columns: df[WEIGHT_COL]=1.0
print("Original cases:",len(df))
display(df[["dataset","k_value",*FEATURE_COLS]].head())


## 2. Build family targets and audit ties
For B6 and Other18 separately, the complete tied-fastest set is recorded. A canonical member is used only for the unexpanded test label; all tied labels are retained for training-only expansion.


In [ ]:
def tied_fastest(row,algos,tol=TIE_TOLERANCE):
    vals={a:float(row[a]) for a in algos}; best=min(vals.values())
    tied=[a for a in algos if vals[a] <= best*(1+tol)]
    return best,tied

family_targets={}; tie_rows=[]
for fam,algos in FAMILIES.items():
    canon={}; tiedmap={}
    for idx,row in df.iterrows():
        best,tied=tied_fastest(row,algos)
        canon[idx]=tied[0]; tiedmap[idx]=tied
        if len(tied)>1:
            tie_rows.append({"Family":fam,"Source Row":idx,"Dataset":row["dataset"],"K Value":row["k_value"],
                             "Best Runtime (s)":best,"Tied Algorithms":" | ".join(tied),
                             "Number Tied":len(tied),"Handling":"Expand only if in training; test remains one row"})
    family_targets[fam]={"canonical":pd.Series(canon),"tied":tiedmap}
tie_audit=pd.DataFrame(tie_rows)
display(tie_audit.head(20)); tie_audit.to_csv(OUTPUT_DIR/"tie_audit.csv",index=False)


## 3. Original-row 80/20 splitting
`strat_y` uses the family target. `strat_y_K` reuses the existing K-aware splitter with target+K. Leakage and duplicate test membership are checked before any expansion.


In [ ]:
def split_original(fam,mode):
    y=family_targets[fam]["canonical"].loc[df.index]; X=df[FEATURE_COLS]
    if mode=="strat_y":
        Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify=y)
    else:
        labels=y.astype(str)+"_k"+df["k_value"].astype(str)
        Xtr,Xte,ytr,yte=_train_test_split_k_aware(
            X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,
            stratify_labels=labels,context_label=f"{fam}_{mode}")
    if set(Xtr.index)&set(Xte.index): raise ValueError(f"{fam}/{mode}: leakage")
    if Xte.index.duplicated().any(): raise ValueError(f"{fam}/{mode}: duplicate test rows")
    return Xtr.index,Xte.index

membership={}; audit=[]
for fam in FAMILIES:
    for mode in SPLITS:
        tr,te=split_original(fam,mode); membership[(fam,mode)]=(tr,te)
        audit.append({"Family":fam,"Split":mode,"Training Cases":len(tr),"Test Cases":len(te),
                      "Train/Test Overlap":len(set(tr)&set(te)),"Duplicate Test Cases":int(pd.Index(te).duplicated().sum())})
split_audit=pd.DataFrame(audit); display(split_audit); split_audit.to_csv(OUTPUT_DIR/"split_audit.csv",index=False)


## 4. Training-only tie expansion
Each tied training instance becomes one supervised row per tied family algorithm, with its sample weight divided across the tied labels. Held-out test data remains one row per original case.


In [ ]:
def prepare(fam,mode):
    tr,te=membership[(fam,mode)]; tied=family_targets[fam]["tied"]; canon=family_targets[fam]["canonical"]
    xr=[]; yr=[]; wr=[]; src=[]
    for idx in tr:
        row=df.loc[idx]; labels=tied[idx]; base=float(row.get(WEIGHT_COL,1.0))
        for a in labels:
            xr.append(row[FEATURE_COLS].to_dict()); yr.append(a); wr.append(base/len(labels)); src.append(idx)
    Xtr=pd.DataFrame(xr,index=src)[FEATURE_COLS]; ytr=pd.Series(yr,index=src); w=pd.Series(wr,index=src)
    Xte=df.loc[te,FEATURE_COLS].copy(); yte=canon.loc[te].copy()
    if set(Xtr.index)&set(Xte.index): raise ValueError("Leakage after expansion")
    return Xtr,Xte,ytr,yte,w

prepared={(f,s):prepare(f,s) for f in FAMILIES for s in SPLITS}
for k,(Xtr,Xte,ytr,yte,w) in prepared.items():
    print(k,"expanded train",len(Xtr),"unique train",Xtr.index.nunique(),"test",len(Xte))


## 5. Train independent RF and XGB flat models
Reuses the existing Scheme-1 flat hyperparameter variants as the closest existing configuration. XGB `num_class` is set to 6 or 18 and its encoder is fixed to the complete family vocabulary.


In [ ]:
def params_for(model,split):
    key="random_forest" if model=="RF" else "xgboost"; pre="rf" if model=="RF" else "xgb"
    return dict(model_cfg["scheme1"][key]["variants"][f"{pre}_{split}"]["hyperparameters"])

models={}; encoders={}
for fam,algos in FAMILIES.items():
    for split in SPLITS:
        Xtr,Xte,ytr,yte,w=prepared[(fam,split)]
        for m in MODELS:
            p=params_for(m,split)
            if m=="RF":
                clf=RandomForestClassifier(**p); clf.fit(Xtr,ytr,sample_weight=w); le=None
            else:
                le=LabelEncoder().fit(algos); p["objective"]="multi:softprob"; p["num_class"]=len(algos)
                clf=XGBClassifier(**p); clf.fit(Xtr,le.transform(ytr),sample_weight=w)
            models[(fam,split,m)]=clf; encoders[(fam,split,m)]=le
            print("Trained",fam,split,m)


## 6. Generate flat held-out and entire-data prediction frames
Produces `prob_*`, predicted algorithm/runtime, family optimum runtime, correctness, and probability-ranked algorithm tuples. These in-memory frames are reused by every downstream analysis.


In [ ]:
def prediction_frame(fam,split,m,scope):
    algos=FAMILIES[fam]; clf=models[(fam,split,m)]; le=encoders[(fam,split,m)]
    _,te=membership[(fam,split)]; idx=te if scope=="test_data" else df.index; X=df.loc[idx,FEATURE_COLS]
    proba=clf.predict_proba(X)
    if m=="RF": classes=list(clf.classes_); pred=clf.predict(X)
    else: classes=list(le.inverse_transform(np.arange(proba.shape[1]))); pred=le.inverse_transform(clf.predict(X).astype(int))
    pos={c:i for i,c in enumerate(classes)}; pc=np.column_stack([proba[:,pos[a]] for a in algos])
    out=df.loc[idx,["dataset","k_value",*FEATURE_COLS,*ALL_ALGOS]].copy()
    out["fastest_algorithm"]=family_targets[fam]["canonical"].loc[idx].values
    out["predicted_algorithm"]=pred
    out["fastest_runtime"]=[min(float(df.loc[i,a]) for a in algos) for i in idx]
    out["predicted_runtime"]=[float(df.loc[i,p]) for i,p in zip(idx,pred)]
    out["predicted_time"]=out["predicted_runtime"]; out["correct"]=out.fastest_algorithm.eq(out.predicted_algorithm)
    out["scheme"]="B6_vs_Other18"; out["family"]=fam; out["model"]=m; out["split_mode"]=split; out["data_scope"]=scope
    for j,a in enumerate(algos): out[f"prob_{a}"]=pc[:,j]
    ranks=[]
    for _,r in out.iterrows():
        ft=float(r.fastest_runtime); z=[(float(r[a])/ft,a,float(r[a]),float(r[f"prob_{a}"])) for a in algos]
        z.sort(key=lambda x:-x[3]); ranks.append(z)
    out["all_predictions_sorted_by_prob"]=ranks
    return out.reset_index().rename(columns={"index":"source_row_index"})

results={(f,s,m,sc):prediction_frame(f,s,m,sc) for f in FAMILIES for s in SPLITS for m in MODELS for sc in ("test_data","entire_data")}
print("Frames:",len(results))


## 7. Metrics and confusion matrices
Reports held-out and entire-data accuracy, macro precision/recall/F1, and weighted precision/recall/F1. Entire-data metrics include training rows and are fit diagnostics.


In [ ]:
metric_rows=[]
for (fam,split,m,scope),r in results.items():
    y=r.fastest_algorithm; p=r.predicted_algorithm; labels=FAMILIES[fam]
    pm,rm,fm,_=precision_recall_fscore_support(y,p,labels=labels,average="macro",zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(y,p,labels=labels,average="weighted",zero_division=0)
    metric_rows.append({"Family":fam,"Split":split,"Model":m,"Scope":scope,"Rows":len(r),
                        "Accuracy":accuracy_score(y,p),"Macro Precision":pm,"Macro Recall":rm,"Macro F1":fm,
                        "Weighted Precision":pw,"Weighted Recall":rw,"Weighted F1":fw})
    cm=confusion_matrix(y,p,labels=labels)
    fig,ax=plt.subplots(figsize=(8,7)); ConfusionMatrixDisplay(cm,display_labels=labels).plot(ax=ax,xticks_rotation=90,values_format="d")
    ax.set_title(f"{fam} | {split} | {m} | {scope}"); fig.tight_layout()
    fig.savefig(OUTPUT_DIR/f"confusion_{fam}_{split}_{m}_{scope}.png",dpi=160,bbox_inches="tight"); plt.show()
metrics_df=pd.DataFrame(metric_rows); display(metrics_df.round(4)); metrics_df.to_csv(OUTPUT_DIR/"metrics_summary.csv",index=False)


## 8. Tolerance metrics
Reuses existing `evaluate_with_tolerance`, `evaluate_time_metrics_with_tol`, and `compute_top_k_metrics` at 0%, 10%, 20%, and 50%.


In [ ]:
tol_rows=[]
for (fam,split,m,scope),r in results.items():
    algos=FAMILIES[fam]
    for tol in TOLERANCES:
        try: acc=evaluate_with_tolerance(r,algos,tol=tol)["accuracy"]
        except Exception: acc=np.nan
        tm=evaluate_time_metrics_with_tol(r,algos,tol=tol,epsilon=EPSILON)
        tk=compute_top_k_metrics(r,algos,tol=tol)
        tol_rows.append({"Family":fam,"Split":split,"Model":m,"Scope":scope,"Tolerance":tol,
                         "Accuracy With Tolerance":acc,"Time Ratio":tm["time_ratio"],
                         "Average Overhead":tm["avg_overhead_per_case"],
                         "Top1 Accuracy":tk["top1_accuracy"],"Top2 Accuracy":tk["top2_accuracy"],"Top3 Accuracy":tk["top3_accuracy"],
                         "Top1 Accuracy With Tolerance":tk["top1_accuracy_with_tolerance"],
                         "Top2 Accuracy With Tolerance":tk["top2_accuracy_with_tolerance"],
                         "Top3 Accuracy With Tolerance":tk["top3_accuracy_with_tolerance"]})
tolerance_df=pd.DataFrame(tol_rows); display(tolerance_df.round(4))
with pd.ExcelWriter(OUTPUT_DIR/"tolerance_metrics.xlsx",engine="openpyxl") as w:
    tolerance_df.to_excel(w,sheet_name="tolerance_summary",index=False)


## 9. Wrong predictions and important failures
Uses the existing `compare_wrong_predictions.filter_tolerance_sheet_for_important_failures()` function on canonical 0%-tolerance wrong predictions.


In [ ]:
failure_frames={}; fs=[]
for key,r in results.items():
    fam,split,m,scope=key; wrong=r[~r.correct].copy()
    imp,stats=filter_tolerance_sheet_for_important_failures(wrong)
    failure_frames[key]=imp
    fs.append({"Family":fam,"Split":split,"Model":m,"Scope":scope,"Total Predictions":len(r),
               "Wrong Predictions":len(wrong),"Important Failures":len(imp),
               "Important / Wrong (%)":round(100*len(imp)/len(wrong),2) if len(wrong) else 0})
failure_summary=pd.DataFrame(fs); display(failure_summary); failure_summary.to_csv(OUTPUT_DIR/"failure_summary.csv",index=False)


## 10. Enriched failure-case workbook
Adds the familiar fastest-algorithm rank, Top-1…Top-6 minimum-time-ratio/candidate columns, and full probability ranking.


In [ ]:
cols=["dataset","k_value","scheme","origin","fastest_algo_rank","fastest_algo_tuple"]
for k in range(1,7): cols += [f"top{k}_min_time_ratio",f"top{k}_candidate"]
cols += ["all_predictions_sorted_by_prob"]

def enrich(r):
    rows=[]
    for _,x in r.iterrows():
        ranking=x.all_predictions_sorted_by_prob; fa=x.fastest_algorithm
        rank=next((i for i,z in enumerate(ranking,1) if z[1]==fa),None)
        q={"dataset":x.dataset,"k_value":x.k_value,"scheme":"B6_vs_Other18","origin":x.family,
           "fastest_algo_rank":rank,"fastest_algo_tuple":f"({float(x.fastest_runtime):.4f}, {fa!r}, {rank})",
           "all_predictions_sorted_by_prob":str(ranking)}
        for k in range(1,7):
            cand=ranking[:min(k,len(ranking))]; best=min(cand,key=lambda z:z[0]) if cand else None
            q[f"top{k}_min_time_ratio"]=round(best[0],4) if best else pd.NA
            q[f"top{k}_candidate"]=f"({best[2]:.4f}, {best[1]!r}, {best[3]:.4f})" if best else pd.NA
        rows.append(q)
    return pd.DataFrame(rows,columns=cols)

enriched={k:enrich(v) for k,v in failure_frames.items()}
with pd.ExcelWriter(OUTPUT_DIR/"failure_analysis.xlsx",engine="openpyxl") as w:
    failure_summary.to_excel(w,sheet_name="summary",index=False)
    for (fam,split,m,scope),e in enriched.items():
        name=f"{fam}_{split}_{m}_{'test' if scope=='test_data' else 'entire'}"[:31]
        e.to_excel(w,sheet_name=name,index=False)


## 11. Old-rule Top-K combinations
For each B Top-i + Other Top-j pair (`i,j=1..3`), each family is independently filtered by its selected `topK_min_time_ratio > 1.20`, then surviving rows are concatenated. A dataset/K failing in both families remains **two rows**, matching the old mixed-analysis rule.


In [ ]:
TH=1.20; combo_frames={}; summary=[]
for split in SPLITS:
  for m in MODELS:
    for scope in ("test_data","entire_data"):
      b=enriched[("B6",split,m,scope)]; o=enriched[("Other18",split,m,scope)]
      for bi in range(1,4):
        for oj in range(1,4):
          bf=b[pd.to_numeric(b[f"top{bi}_min_time_ratio"],errors="coerce").gt(TH)].copy()
          of=o[pd.to_numeric(o[f"top{oj}_min_time_ratio"],errors="coerce").gt(TH)].copy()
          bf["source_rule"]=f"B{bi}+O{oj}"; of["source_rule"]=f"B{bi}+O{oj}"
          z=pd.concat([bf,of],ignore_index=True).sort_values(["dataset","k_value","origin"],kind="stable")
          kb=set(zip(bf.dataset.astype(str),bf.k_value)) if len(bf) else set()
          ko=set(zip(of.dataset.astype(str),of.k_value)) if len(of) else set()
          shared=kb&ko; unique=kb^ko; combo_frames[(split,m,scope,bi,oj)]=z
          summary.append({"Split":split,"Model":m,"Scope":scope,"Combination":f"B{bi}+O{oj}",
                          "Failure Rows After Top-K":len(z),"Shared Dataset/K Cases":len(shared),
                          "Rows From Shared Cases":2*len(shared),"Unique Dataset/K Cases":len(unique),
                          "B6 Remaining Rows":len(bf),"Other18 Remaining Rows":len(of)})
combo_summary=pd.DataFrame(summary); display(combo_summary); combo_summary.to_csv(OUTPUT_DIR/"topk_combination_summary.csv",index=False)
for split in SPLITS:
  for m in MODELS:
    for scope in ("test_data","entire_data"):
      path=OUTPUT_DIR/f"topk_{split}_{m}_{scope}.xlsx"
      with pd.ExcelWriter(path,engine="openpyxl") as w:
        combo_summary[(combo_summary.Split==split)&(combo_summary.Model==m)&(combo_summary.Scope==scope)].to_excel(w,sheet_name="summary",index=False)
        for bi in range(1,4):
          for oj in range(1,4):
            combo_frames[(split,m,scope,bi,oj)].to_excel(w,sheet_name=f"B{bi}_O{oj}",index=False)


## 12. Slide-ready tables
Primary tables: held-out model quality, held-out wrong/important failures, and held-out Top-K remaining failures.


In [ ]:
slide_metrics=metrics_df[metrics_df.Scope=="test_data"][["Family","Split","Model","Rows","Accuracy","Macro F1","Weighted F1"]].copy()
for c in ["Accuracy","Macro F1","Weighted F1"]: slide_metrics[c]=(100*slide_metrics[c]).round(2)
slide_fail=failure_summary[failure_summary.Scope=="test_data"].copy()
slide_topk=combo_summary[combo_summary.Scope=="test_data"].copy()
display(slide_metrics); display(slide_fail); display(slide_topk)
slide_metrics.to_csv(OUTPUT_DIR/"slide_metrics.csv",index=False)
slide_fail.to_csv(OUTPUT_DIR/"slide_failures.csv",index=False)
slide_topk.to_csv(OUTPUT_DIR/"slide_topk.csv",index=False)


# Step-by-step implementation summary

- **Setup:** reuses `features.py`, `data_splits.py`, `train.py` configuration loading, `metrics.py`, `evaluate.py`, and the existing important-failure filter.
- **Data:** loads and preprocesses the canonical runtime dataset.
- **Families:** derives the six B algorithms from `ROUTER_CLASSB_ALGOS`; the other 18 are its complement.
- **Ties:** audits family-specific ties. Original rows are split first; tied labels are expanded only inside training with divided weight; test remains unexpanded.
- **Splits:** runs both `strat_y` and `strat_y_K` with explicit leakage/duplicate checks.
- **Training:** trains independent B6 and Other18 flat RF and XGB models.
- **Predictions:** generates held-out and entire-data probability rankings in memory.
- **Metrics:** reports classification metrics and confusion matrices.
- **Tolerance:** evaluates 0/10/20/50% tolerance using existing evaluation functions.
- **Important failures:** applies the existing repository filter to wrong predictions.
- **Failure analysis:** creates the familiar Top-1…Top-6 enriched failure layout.
- **Top-K combinations:** applies the old independent filtering rule for B1–B3 × O1–O3; shared failures remain two rows.
- **Slides:** exports compact metrics, failure, and Top-K summary CSVs.

Only required analysis artifacts are written under `restructured_pipeline/outputs/reports/B6_vs_Other18_independent_flat/`; no new source/config file is introduced.
